# Fine-tuning de baseline: ViT-B/16 (2021) — supervisado (Attention-Transformer)

Segunda etapa del banco (Objetivo 2): el mismo backbone del test congelado,
ahora **fine-tuning end-to-end** sobre **ISIC 2019 (multiclase, 8 clases del challenge oficial)** (cabeza lineal +
backbone completo). Mismo split anti-fuga y mismas metricas que el banco
congelado -> comparacion directa congelado-vs-fine-tuned por modelo.

Protocolo identico para los 8 modelos: AdamW (LR backbone 5e-5, LR cabeza
5e-4), weighted cross-entropy, coseno, AMP fp16, mejor epoca por balanced
accuracy de validacion.

**Como correr en Kaggle**
1. Add Input: `ISIC 2019` (andrewmvd/isic-2019), el release con ground truth multiclase.
2. Settings: GPU T4 + Internet ON (descarga de pesos).
3. Primero `SMOKE = True` (1 epoca con subconjunto; numeros NO validos).
4. Luego `SMOKE = False` -> Save & Run All (~1-2 h en T4 segun modelo).
5. Si hay OOM (vgg19 / vit_b16 / dinov2_vitb14): bajar `BATCH` a 16.

Resultado: `/kaggle/working/results_vit_b16_ft_isic2019.json` +
checkpoint `ft_vit_b16_ft_isic2019.pth` + tabla impresa.


In [ ]:
MODEL = "vit_b16"   # generado por scripts/build_finetune_tests.py - no editar a mano
DATASET = "isic2019"
JSON_SUFFIX = "_ft_isic2019"
SMOKE = True        # True = 1 epoca con subconjunto (numeros NO validos)
BATCH = 32          # fine-tuning necesita memoria para gradientes; OOM -> 16
EPOCHS = 8
LR_BACKBONE = 5e-5
LR_HEAD = 5e-4
SEED = 42
KNN_K = 20          # sin uso aqui; se mantiene por compatibilidad de celdas
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
print("modelo:", MODEL, "| device:", device, "| smoke:", SMOKE)


In [ ]:
# --- ISIC 2019 (andrewmvd/isic-2019) desde /kaggle/input: tarea MULTICLASE de
# 8 clases, la misma del challenge oficial. Es el UNICO banco cuya tarea coincide
# con el ancla de Gessert (0.636 de balanced accuracy en el test oficial), asi
# que es el unico comparable de frente contra ella.
#
# LIMITE que viaja con estos numeros: este release NO trae columna de paciente
# (y el patient_id del RESIZED es sintetico), asi que NO hay split anti-fuga por
# paciente. El split es estratificado simple. Dos imagenes del mismo paciente
# pueden caer en train y en val, lo que sesga las metricas hacia arriba. Por eso
# este banco NO es comparable 1:1 con HAM10000 ni con ISIC 2020, que si agrupan
# por lesion_id y patient_id reales. ---
from sklearn.model_selection import StratifiedKFold

root = Path("/kaggle/input")
gt_csv = next(root.glob("**/ISIC_2019_Training_GroundTruth.csv"))
meta = pd.read_csv(gt_csv)
id_col = meta.columns[0]                      # "image"
# UNK sale en cero en todo el release: conservarla inventaria una novena clase
# que nunca ocurre y hundiria todas las metricas macro por un factor 9/8.
onehot = meta.drop(columns=[id_col] + [c for c in ("UNK",) if c in meta.columns])
classes = list(onehot.columns)
N = len(classes)
labels = onehot.idxmax(axis=1)                # one-hot -> nombre de clase
cls2idx = {c: i for i, c in enumerate(classes)}
paths = {p.stem: p for p in root.glob("**/*.jpg")}
meta = meta.assign(_label=labels)
meta = meta[meta[id_col].astype(str).isin(paths)].reset_index(drop=True)
id2label = {str(r): cls2idx[l] for r, l in zip(meta[id_col], meta._label)}
print("clases:", classes); print(meta._label.value_counts().to_string())

ids = meta[id_col].astype(str).tolist()
y = np.array([id2label[i] for i in ids])
tr_idx, va_idx = next(StratifiedKFold(5, shuffle=True, random_state=SEED)
                      .split(ids, y))
tr_ids = [ids[i] for i in tr_idx]; va_ids = [ids[i] for i in va_idx]
assert not set(tr_ids) & set(va_ids), "una imagen quedo en train y en val"

if SMOKE:  # subconjunto estratificado: humo en minutos
    rng = np.random.default_rng(SEED)
    def subsample(id_list, frac, min_pc):
        yy = np.array([id2label[i] for i in id_list]); keep = []
        for c in range(N):
            idx = np.flatnonzero(yy == c)
            take = max(min_pc, int(len(idx) * frac))
            keep.extend(rng.choice(idx, size=min(take, len(idx)), replace=False))
        return [id_list[i] for i in sorted(keep)]
    tr_ids = subsample(tr_ids, 0.08, 12); va_ids = subsample(va_ids, 0.08, 6)

y_tr = np.array([id2label[i] for i in tr_ids])
y_va = np.array([id2label[i] for i in va_ids])
SPLIT_DESC = ("StratifiedKFold estratificado simple SIN grupo de paciente "
              "(el release no trae patient_id) - riesgo de fuga entre imagenes "
              "del mismo paciente, metricas con sesgo optimista")
print(f"train={len(tr_ids)} val={len(va_ids)} | SIN agrupacion por paciente")


In [ ]:
# --- Backbone CONGELADO (nunca se entrena). Fuente de pesos: Hugging Face Hub
# via timm (decision del equipo con la Dra. Karen, 2026-08-28). Los tags
# .tv_in1k / .tv2_in1k de timm sirven LOS MISMOS tensores que torchvision ->
# resnet50 sigue comparable con las corridas previas del banco. OJO: los mismos
# tensores NO implican el mismo punto de extraccion — ver el caso vgg19 abajo.
# Excepciones documentadas (sin pesos oficiales en HF): alexnet (torchvision)
# y simsiam_r50 (checkpoint oficial de FAIR). model.eval() SIEMPRE:
# requires_grad=False no congela BatchNorm. ---
RES, INTERP = 224, transforms.InterpolationMode.BILINEAR

# Ancho de features esperado: GAP del ultimo bloque conv (CNN) o token CLS
# (ViT). Se asserta abajo — un cambio de punto de extraccion rompe la
# comparabilidad del banco y tiene que fallar fuerte, no en silencio.
FEATURE_DIMS = {
    "alexnet": 256, "vgg19": 512, "efficientnet_b0": 1280, "resnet50": 2048,
    "vit_b16": 768, "simsiam_r50": 2048, "dinov2_vits14": 384,
    "dinov2_vitb14": 768,
}

TIMM_HF_TAGS = {
    "vgg19": "vgg19.tv_in1k",                       # = torchvision IMAGENET1K_V1
    "efficientnet_b0": "efficientnet_b0.ra_in1k",   # receta original de timm
    "resnet50": "resnet50.tv2_in1k",                # = torchvision IMAGENET1K_V2
    "vit_b16": "vit_base_patch16_224.augreg_in1k",  # supervisado SOLO in1k (fair)
    "dinov2_vits14": "vit_small_patch14_dinov2.lvd142m",
    "dinov2_vitb14": "vit_base_patch14_dinov2.lvd142m",
}

try:
    import timm
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

def gap(features):
    return nn.Sequential(features, nn.AdaptiveAvgPool2d(1), nn.Flatten(1))

def load_backbone(name):
    global RES, INTERP
    if name == "alexnet":
        # Sin pesos oficiales en HF Hub: se mantiene torchvision, documentado.
        from torchvision.models import AlexNet_Weights, alexnet
        return gap(alexnet(weights=AlexNet_Weights.IMAGENET1K_V1).features)
    if name == "simsiam_r50":
        # Checkpoint oficial de FAIR (no publicado en HF). Pendiente: mirror
        # en el repo HF del equipo para fijar revision.
        from torchvision.models import resnet50
        url = ("https://dl.fbaipublicfiles.com/simsiam/models/100ep-256bs/"
               "pretrain/checkpoint_0099.pth.tar")
        sd = torch.hub.load_state_dict_from_url(url, map_location="cpu")["state_dict"]
        sd = {k.replace("module.encoder.", ""): v for k, v in sd.items()
              if k.startswith("module.encoder") and "fc" not in k}
        m = resnet50(weights=None); m.fc = nn.Identity()
        missing, unexpected = m.load_state_dict(sd, strict=False)
        assert not unexpected, f"claves inesperadas: {unexpected[:5]}"
        return m
    tag = TIMM_HF_TAGS[name]
    if name == "efficientnet_b0":
        INTERP = transforms.InterpolationMode.BICUBIC  # su distribucion de entrenamiento
    kwargs = {"img_size": RES} if "patch14" in tag else {}
    m = timm.create_model(tag, pretrained=True, num_classes=0, **kwargs)
    if name == "vgg19":
        # num_classes=0 deja las pre-logits de 4096 (fc6/fc7 de la cabeza); el
        # banco lee toda CNN en el GAP de su ultimo bloque conv.
        return gap(m.features)
    return m

model = load_backbone(MODEL)
for p in model.parameters():
    p.requires_grad = False
model.eval().to(device)
with torch.no_grad():
    DIM = model(torch.zeros(1, 3, RES, RES, device=device)).shape[-1]
trainable = sum(p.requires_grad for p in model.parameters())
assert trainable == 0, "el backbone NO esta congelado"
assert DIM == FEATURE_DIMS[MODEL], (
    f"{MODEL}: se extrajeron {DIM} dims, el protocolo fija {FEATURE_DIMS[MODEL]}. "
    "El punto de extraccion cambio; la corrida NO es comparable con el banco.")
print(f"{MODEL}: dim={DIM} res={RES} interp={INTERP} entrenables=0")


In [ ]:
# --- Metricas de literatura (identicas al banco congelado) ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import normalize

def bootstrap_ci(y_true, pred, n_resamples=1000):
    # IC 95 percentil sobre remuestreo del set de validacion. La diferencia
    # entre "0.629" y "0.629 [0.61, 0.65]" es lo primero que pregunta un jurado.
    rng = np.random.default_rng(SEED)
    idx_all = np.arange(len(y_true)); ba, f1v = [], []
    for _ in range(n_resamples):
        idx = rng.choice(idx_all, size=len(idx_all), replace=True)
        ba.append(balanced_accuracy_score(y_true[idx], pred[idx]))
        f1v.append(f1_score(y_true[idx], pred[idx], average="macro"))
    pct = lambda v: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]
    return {"balanced_accuracy": pct(ba), "macro_f1": pct(f1v)}

def ece(y_true, proba, bins=15):
    conf = proba.max(1); pred = proba.argmax(1)
    ok = (pred == y_true).astype(float); edges = np.linspace(0, 1, bins + 1); tot = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1]) if i else (conf >= 0) & (conf <= edges[1])
        if m.any():
            tot += m.mean() * abs(conf[m].mean() - ok[m].mean())
    return float(tot)

def all_metrics(y_true, proba):
    pred = proba.argmax(1)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {
            "accuracy": accuracy_score(y_true, pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
            "macro_f1": f1_score(y_true, pred, average="macro"),
            "macro_precision": precision_score(y_true, pred, average="macro", zero_division=0),
            "macro_recall": recall_score(y_true, pred, average="macro", zero_division=0),
            # binario: sklearn exige la proba de la clase positiva (1-D);
            # multiclase: OvR macro sobre la matriz completa.
            "auc_ovr_macro": (roc_auc_score(y_true, proba[:, 1]) if N == 2 else
                              roc_auc_score(y_true, proba, multi_class="ovr",
                                            average="macro", labels=list(range(N)))),
            "ece": ece(y_true, proba),
            "recall_por_clase": {classes[c]: float(recall_score(
                y_true, pred, labels=[c], average="macro", zero_division=0))
                for c in range(N)},
            "precision_por_clase": {classes[c]: float(precision_score(
                y_true, pred, labels=[c], average="macro", zero_division=0))
                for c in range(N)},
            "f1_por_clase": {classes[c]: float(f1_score(
                y_true, pred, labels=[c], average="macro", zero_division=0))
                for c in range(N)},
            "matriz_confusion": confusion_matrix(
                y_true, pred, labels=list(range(N))).tolist(),
            "ic95_bootstrap": bootstrap_ci(y_true, pred),
        }



In [ ]:
# --- Fine-tuning end-to-end: se descongela el backbone y se agrega cabeza
# lineal. El assert de la celda anterior verifico el estado CONGELADO de
# carga; aqui empieza el entrenamiento real. ---
import copy
from sklearn.metrics import balanced_accuracy_score

EPOCHS_RUN = 1 if SMOKE else EPOCHS
for p in model.parameters():
    p.requires_grad = True

# La celda anterior imprime "entrenables=0" porque comparte codigo con el banco
# CONGELADO. Ese numero ya no vale aqui: se re-imprime despues de descongelar
# para que el log no haga dudar de si esto entreno de verdad.
_entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _entrenables > 0, "el backbone sigue congelado: esto NO es fine-tuning"
print(f"{MODEL}: backbone DESCONGELADO, entrenables={_entrenables:,}")

class FTNet(nn.Module):
    def __init__(self, backbone, dim, n_cls):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(dim, n_cls)
    def forward(self, x):
        return self.head(self.backbone(x))

net = FTNet(model, DIM, N).to(device)

tf_train = transforms.Compose([
    transforms.Resize(int(RES * 1.14), interpolation=INTERP),
    transforms.RandomCrop(RES),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
tf_eval = transforms.Compose([
    transforms.Resize(int(RES * 1.14), interpolation=INTERP),
    transforms.CenterCrop(RES),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class FTDataset(Dataset):
    def __init__(self, id_list, tf_):
        self.ids, self.tf = id_list, tf_
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        img = Image.open(paths[self.ids[i]]).convert("RGB")
        return self.tf(img), id2label[self.ids[i]]

dl_tr = DataLoader(FTDataset(tr_ids, tf_train), batch_size=BATCH, shuffle=True,
                   num_workers=2, pin_memory=True, drop_last=True)
dl_va = DataLoader(FTDataset(va_ids, tf_eval), batch_size=BATCH, shuffle=False,
                   num_workers=2, pin_memory=True)

counts = np.bincount(y_tr, minlength=N).astype(float)
class_w = torch.tensor(counts.sum() / (N * counts), dtype=torch.float32,
                       device=device)
criterion = nn.CrossEntropyLoss(weight=class_w)
optimizer = torch.optim.AdamW([
    {"params": net.backbone.parameters(), "lr": LR_BACKBONE},
    {"params": net.head.parameters(), "lr": LR_HEAD},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS_RUN * len(dl_tr)))
use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# La curva viaja DENTRO del JSON, no solo en el log de Kaggle (que expira).
# Sin esto, una corrida truncada se ve igual que una convergida: dinov2_vitb14
# dio 0.533 en HAM10000, por debajo de su propio probe congelado, porque ocho
# epocas no le alcanzaron - y esa evidencia no estaba en ningun lado salvo el log.
HISTORIA = []
best_ba, best_sd, best_epoch = -1.0, None, 0
t0 = time.time()
for epoch in range(EPOCHS_RUN):
    net.train(); running, seen = 0.0, 0
    for xb, yb in dl_tr:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = criterion(net(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item() * len(yb); seen += len(yb)
    net.eval(); preds = []
    with torch.no_grad():
        for xb, _ in dl_va:
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = net(xb.to(device, non_blocking=True))
            preds.append(out.float().argmax(1).cpu().numpy())
    ba = balanced_accuracy_score(y_va, np.concatenate(preds))
    train_loss = running / max(1, seen)
    HISTORIA.append({"epoca": epoch + 1, "train_loss": float(train_loss),
                     "val_bal_acc": float(ba)})
    if ba > best_ba:
        best_ba, best_sd, best_epoch = ba, copy.deepcopy(net.state_dict()), epoch + 1
    print(f"epoca {epoch + 1}/{EPOCHS_RUN} loss={train_loss:.4f} "
          f"val_bal_acc={ba:.4f} mejor={best_ba:.4f} ({time.time() - t0:.0f}s)")

net.load_state_dict(best_sd)  # evaluacion final sobre la MEJOR epoca
# Si la mejor epoca es la ultima, el entrenamiento se corto mientras todavia
# mejoraba: el numero es un piso, no un techo. Queda en el JSON para que
# cualquiera lo detecte sin leer logs.
TRUNCADA = best_epoch == EPOCHS_RUN
print(f"fine-tuning listo: mejor bal_acc={best_ba:.4f} en epoca {best_epoch}"
      f"/{EPOCHS_RUN} ({time.time() - t0:.0f}s)")
if TRUNCADA:
    print("AVISO: la mejor epoca es la ultima -> la corrida quedo TRUNCADA, "
          "el modelo seguia mejorando. NO comparable con una corrida convergida.")


In [ ]:
# --- Evaluacion final del modelo fine-tuned (mismas metricas del banco) ---
net.eval(); probs = []
with torch.no_grad():
    for xb, _ in dl_va:
        out = net(xb.to(device, non_blocking=True))
        probs.append(torch.softmax(out.float(), 1).cpu().numpy())
proba_va = np.concatenate(probs)

results = {}
results["finetune"] = all_metrics(y_va, proba_va)

dest = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
ckpt = dest / f"ft_{MODEL}{JSON_SUFFIX}.pth"
torch.save({"modelo": MODEL, "dataset": DATASET, "smoke": SMOKE,
            "clases": classes, "state_dict": net.state_dict()}, ckpt)
print("checkpoint:", ckpt)


In [ ]:
# --- Reporte final + JSON ---
out = {
    "modelo": MODEL, "dataset": DATASET, "smoke": SMOKE,
    "n_train": len(tr_ids), "n_val": len(va_ids),
    "dim_features": int(DIM), "resolucion": RES, "semilla": SEED,
    "split": SPLIT_DESC,
    "probes": results,
    "mejor_epoca": best_epoch, "epocas_corridas": EPOCHS_RUN,
    "truncada": TRUNCADA,
    "historia": HISTORIA,
    "protocolo": {"epocas": EPOCHS_RUN, "batch": BATCH,
                  "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD,
                  "amp": use_amp},
    "generador_sha": "5a982148c03936ca",
}
dest = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
(dest / f"results_{MODEL}{JSON_SUFFIX}.json").write_text(json.dumps(out, indent=2))

hdr = ["probe", "bal_acc", "macro_f1", "precision", "recall", "auc", "acc", "ece"]
print(("ATENCION: numeros de SMOKE, no validos\n" if SMOKE else "") + "-" * 86)
print(f"{MODEL} | val n={len(va_ids)}")
print("{:<11}{:>9}{:>10}{:>11}{:>9}{:>8}{:>8}{:>8}".format(*hdr))
for probe, m in results.items():
    print("{:<11}{:>9.4f}{:>10.4f}{:>11.4f}{:>9.4f}{:>8.4f}{:>8.4f}{:>8.4f}".format(
        probe, m["balanced_accuracy"], m["macro_f1"], m["macro_precision"],
        m["macro_recall"], m["auc_ovr_macro"], m["accuracy"], m["ece"]))
print("-" * 86)
ci = results["finetune"]["ic95_bootstrap"]
print("finetune IC95 bootstrap: bal_acc {:.3f}-{:.3f} | F1 macro {:.3f}-{:.3f}".format(
    *ci["balanced_accuracy"], *ci["macro_f1"]))
if DATASET == "ham10000":
    print("ancla de literatura: ganador ISIC 2019 (Gessert) = 0.636 balanced accuracy")
print("guardado:", dest / f"results_{MODEL}{JSON_SUFFIX}.json")


In [ ]:
# --- Trazabilidad MLflow (no fatal: si falta, el test ya corrio igual) ---
# Mismo criterio que v6 (docs/16): backend de archivos, sin servidor.
# En Kaggle: los runs quedan en /kaggle/working/mlruns (descargables como Output).
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"  # MLflow 3.x: file store pide opt-in
try:
    import mlflow
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow-skinny"],
                   check=False)
    try:
        import mlflow
    except ImportError:
        mlflow = None
        print("AVISO: mlflow no disponible - el test corrio igual; JSON ya guardado")

if mlflow is not None:
    mlflow.set_tracking_uri(f"file:{dest / 'mlruns'}")
    mlflow.set_experiment(f"finetune-{DATASET}")
    with mlflow.start_run(run_name=f"{MODEL}{'-smoke' if SMOKE else ''}"):
        mlflow.log_params({
            "modelo": MODEL, "dataset": DATASET, "smoke": SMOKE, "semilla": SEED,
            "resolucion": RES, "dim_features": int(DIM), "n_train": len(tr_ids),
            "n_val": len(va_ids), "knn_k": KNN_K, "split": SPLIT_DESC,
        })
        for probe, m in results.items():
            for name, val in m.items():
                if isinstance(val, (int, float)):
                    mlflow.log_metric(f"{probe}.{name}", val)
        mlflow.log_artifact(str(dest / f"results_{MODEL}{JSON_SUFFIX}.json"))
    print("MLflow: run registrado en", dest / "mlruns")
    print("ver UI local: MLFLOW_ALLOW_FILE_STORE=true mlflow ui "
          "--backend-store-uri file:" + str(dest / "mlruns"))
